In [ ]:
import matplotlib.pyplot as plt

def image_show(imagem):
    """
    Exibe uma imagem usando matplotlib.
    :param imagem: Objeto de imagem PIL
    """
    plt.imshow(imagem)
    plt.show()


In [ ]:
import torchvision.models as models 
import torch

def resnet_50(num_class):
    # Create instace of resnet50
    model = models.resnet50(weights = "ResNet50_Weights.DEFAULT")
    # Take total features from the last layer
    num_features = model.fc.in_features
    # Create my own layer with num_class do i want
    model.fc = torch.nn.Linear(num_features, num_class)
    # Multi category cross entropy(Funcao de erro)
    # Binario e multi categorico
    return model    

In [11]:
import os 
from time import sleep
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report
from tqdm import tqdm
from utils.customDataset import CustomDataset
from utils.loadDataset import TrainDatasetImplemetation
# 
# from models.resnet import resnet_50 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   

print("Dispositivo utilizado: ", device)

Dispositivo utilizado:  cpu


In [12]:
data_set_path = "../imagens"    

image_size = (224, 224)
batch_size = 32
data_set = TrainDatasetImplemetation(data_set_path, image_size, batch_size)
train_data, test_data = data_set.load_data()

classes name: ['com_animal', 'sem_animal']
classes name: []
no of samples in train dataset 6835
no of samples in test dataset 0


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# Define binary classficiation!
num_classes = 2
# Create the model implementation
model = resnet_50(num_classes)
# Pass model to the device
model = model.to(device)
# Train model 
model.train()

In [ ]:
# Define the learnin rate
learning_rate = 1e-3
# Number of epochs
num_epochs = 30
'''
    Define the loss function for multi-class classification.
    CrossEntropyLoss combines LogSoftmax and Negative Log-Likelihood Loss (NLLLoss) in one single class.
    It expects raw, unnormalized logits as input and automatically applies softmax.
    The target should contain class indices (e.g., 0, 1, 2...) corresponding to the correct class.

    Internally, for each prediction:
    1. Applies softmax to convert logits into probabilities.
    2. Takes the log of the probability for the correct class.
    3. Applies the negative log to calculate the loss.
    Intuition:
    - If the model gives high probability to the correct class → low loss (good).
    - If the model gives low probability to the correct class → high loss (bad).
    Used for classification problems with two or more classes.
'''
criterion = nn.CrossEntropyLoss()
'''
    Define the optimizer to update the model's parameters during training.
    Adam (Adaptive Moment Estimation) is an optimization algorithm that combines
    the benefits of AdaGrad and RMSProp. It adapts the learning rate for each parameter
    using estimates of the first and second moments of the gradients.

    Arguments:
    - model.parameters(): passes all trainable parameters of the model to the optimizer.
    - lr=learning_rate: sets the initial learning rate for updating the weights.

    Adam is widely used because it typically converges faster and requires less tuning
    of the learning rate compared to traditional stochastic gradient descent (SGD).
'''
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
def get_model_accuracy(test_data, model):
    '''
        Evaluate the model's performance on the test data and print a classification report.

        Arguments:
        - test_data: a DataLoader containing the test dataset.
        - model: the trained model to evaluate.
    '''
    # Initialize arrays to store predicted and true labels
    y_pred = np.zeros(0)
    y_true = np.zeros(0)
    
    # Set the model to evaluation mode (disables dropout, batch norm updates, etc.)
    model.eval()
    # Disable gradient computation to save memory and improve performance
    with torch.no_grad():
        for images_batch, y_true_batch in test_data:

            images_batch = images_batch.to(device)

            scores = model(images_batch)
            _, y_pred_batch = scores.max(1)
            y_pred_batch = y_pred_batch.cpu()

            y_pred = np.concatenate((y_pred, y_pred_batch))
            y_true = np.concatenate((y_true, y_true_batch))

    model.train()

    report = classification_report(y_true, y_pred)
    print(report)

In [ ]:
for epoch in range(1, num_epochs + 1):
    # Create a tqdm progress bar for visualizing training progress per batch
    pbar_batch = tqdm(train_data, unit="batch")
    losses = []
    # Iterate over each batch of data
    for data in pbar_batch:
        # Set the description of the progress bar to show current epoch

        pbar_batch.set_description(f"Epoch {epoch}")
        # Unpacking and sending to GPU
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        # raw scores before softmax
        scores = model(images)
        # We calculate the loss using criterion (CrossEntropyLoss).
        loss = criterion(scores, labels)
        losses.append(loss.item())
        cost = sum(losses)/len(losses)

        # Backward Pass and Weight Update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Updates the progress bar
        pbar_batch.set_postfix(loss=cost)
        sleep(0.1)
    '''
        Every 5 epochs, the model is evaluated with the 
        test data (test_data) using the get_model_accuracy() function.
    '''
    if (epoch % 5) == 0:
        get_model_accuracy(test_data, model)
        sleep(0.1)


print("model training complete")

In [ ]:
'''
    saves the trained model weights to disk, creating the directory 
    if it does not already exist.
'''
model_save_path = "weights/resnet"
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

torch.save(model.state_dict(), model_save_path + "/resnet_model_checkpoint.pth")